In [1]:
import os
import re
import csv
import string
from pathlib import Path
import pandas as pd
import numpy as np

# =====================================================================
# CONFIGURATION
# =====================================================================
SOURCE_FILENAMES = [
    "detik.csv",
    "kumparan.csv",
    "tempo.csv",
    "mojok.csv",
    "tirto.csv",
    "synthetic_imposter_dataset.csv",
]

CANDIDATE_BASE_PATHS = [
    "/kaggle/input/datasets/xandertrevor/stylometry/raw/",
    "/kaggle/input/stylometry/raw/",
    "/kaggle/input/",
    "./data/raw/",
    "../data/raw/",
    ".",
]

WINDOWS_FALLBACKS = {
    "detik.csv": r"d:\StyloGuard (Branch)\StyloGuard\backend\data\raw\detik.csv",
    "kumparan.csv": r"d:\StyloGuard (Branch)\StyloGuard\backend\data\raw\kumparan.csv",
    "mojok.csv": r"d:\StyloGuard (Branch)\StyloGuard\backend\data\raw\mojok.csv",
    "tempo.csv": r"d:\StyloGuard (Branch)\StyloGuard\backend\data\raw\tempo.csv",
    "tirto.csv": r"d:\StyloGuard (Branch)\StyloGuard\backend\data\raw\tirto.csv",
    "synthetic_imposter_dataset.csv": r"d:\StyloGuard (Branch)\StyloGuard\backend\data\raw\synthetic_imposter_dataset.csv",
}

# Organizations/non-person authors to drop explicitly (case-insensitive)
ORGANIZATIONS_TO_DROP = {
    "antara", "redaksi", "kumparannews", "kumparantech", "reuters", "tangsel_update"
}

# Minimum article count threshold for single-word authors
SINGLE_WORD_FREQ_THRESHOLD = 30

# Output file path
OUTPUT_FILENAME = "merged_dataset.csv"

# =====================================================================
# UTILITIES
# =====================================================================
def find_file(filename):
    """Search for the dataset file across Kaggle and local paths."""
    for base in CANDIDATE_BASE_PATHS:
        path = Path(base) / filename
        if path.exists():
            return str(path)

    # Recursive search under /kaggle/input if present
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        matches = sorted(kaggle_root.rglob(filename), key=lambda p: len(str(p)))
        if matches:
            return str(matches[0])

    # Fallback to local Windows paths
    fallback = Path(WINDOWS_FALLBACKS.get(filename, ""))
    if fallback.exists():
        return str(fallback)

    return None

def read_csv_safely(path):
    """Robust CSV reading supporting mixed encodings and bad lines."""
    try:
        return pd.read_csv(path, encoding="utf-8-sig", encoding_errors="replace")
    except TypeError:
        return pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        try:
            return pd.read_csv(
                path,
                encoding="utf-8-sig",
                encoding_errors="replace",
                engine="python",
                on_bad_lines="skip",
            )
        except TypeError:
            return pd.read_csv(path, encoding="utf-8-sig", engine="python", on_bad_lines="skip")

def clean_text(text):
    """Normalize whitespace and convert text to string."""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_author_name(author):
    """Normalize whitespace and strip '(Kontributor)' suffix."""
    author = str(author)
    author = re.sub(r"\s+", " ", author).strip()
    # Strip (Kontributor) suffix case-insensitively
    author = re.sub(r"\s*\(\s*kontributor\s*\)\s*$", "", author, flags=re.IGNORECASE).strip()
    return author

# =====================================================================
# PROCESSING
# =====================================================================
def main():
    print("Starting StyloGuard Dataset Merging and Cleaning...")
    frames = []

    for filename in SOURCE_FILENAMES:
        path = find_file(filename)
        if path is None:
            print(f"⚠️ Warning: Could not find file {filename}. Skipping.")
            continue

        print(f"📖 Reading {filename} from {path}...")
        df = read_csv_safely(path)
        
        # Standardize columns to lowercase and stripped
        df.columns = [str(c).strip().lower() for c in df.columns]
        
        if "text" not in df.columns or "author" not in df.columns:
            print(f"❌ Error: Required columns 'text' and 'author' not found in {filename}.")
            continue

        df = df.copy()
        
        # Add tracking source
        source_name = filename.replace(".csv", "")
        df["source"] = source_name

        # Align columns
        for col in ["title", "date", "category", "url"]:
            if col not in df.columns:
                df[col] = ""
                
        # Fill missing values
        df["text"] = df["text"].fillna("")
        df["author"] = df["author"].fillna("Unknown")
        df["title"] = df["title"].fillna("Untitled")
        df["date"] = df["date"].fillna("")
        df["category"] = df["category"].fillna("uncategorized")
        df["url"] = df["url"].fillna("")

        # Basic filtering and cleaning
        df["text"] = df["text"].map(clean_text)
        df["author"] = df["author"].map(normalize_author_name)
        
        # Filter rows
        df = df[df["text"].str.strip().ne("")]
        df = df[df["author"].str.lower().ne("author")]
        df = df[df["text"].str.split().str.len() >= 50]

        # Handle synthetic dataset explicitly
        if filename == "synthetic_imposter_dataset.csv":
            df["author"] = "AI"

        # Retain necessary columns
        df = df[["author", "text", "title", "date", "category", "url", "source"]]
        frames.append(df)

    if not frames:
        raise RuntimeError("No valid datasets were loaded. Please check your file paths.")

    # Combine all frames
    df_merged = pd.concat(frames, ignore_index=True)
    print(f"\nInitial merged rows: {len(df_merged)}")

    # 1. Clean author names and drop explicit organizations
    df_merged["author"] = df_merged["author"].map(normalize_author_name)
    df_merged = df_merged[~df_merged["author"].str.lower().isin(ORGANIZATIONS_TO_DROP)]
    print(f"Rows after dropping explicit organizations: {len(df_merged)}")

    # 2. Strict deduplication on (author, text) pairs
    df_merged = df_merged.drop_duplicates(subset=["author", "text"]).reset_index(drop=True)
    print(f"Rows after (author, text) deduplication: {len(df_merged)}")

    # 3. Filter out single-word authors with low article frequencies
    # Count author occurrences in the entire dataset
    author_counts = df_merged["author"].value_counts()
    
    def keep_author(author_name):
        if author_name == "AI":
            return True
        # Check if name is single word (no whitespace)
        is_single_word = " " not in author_name
        if is_single_word:
            freq = author_counts.get(author_name, 0)
            if freq < SINGLE_WORD_FREQ_THRESHOLD:
                return False
        return True

    df_merged = df_merged[df_merged["author"].map(keep_author)].reset_index(drop=True)
    print(f"Rows after filtering low-frequency single-word authors: {len(df_merged)}")

    # Final whitespace cleanup on text and author
    df_merged["text"] = df_merged["text"].str.strip()
    df_merged["author"] = df_merged["author"].str.strip()

    # Save to disk
    df_merged.to_csv(OUTPUT_FILENAME, index=False, encoding="utf-8-sig")
    print(f"\n✅ Success! Merged dataset saved as '{OUTPUT_FILENAME}'")
    
    # Print rich analytics summary
    print("\n" + "=" * 50)
    print("             MERGED DATASET SUMMARY")
    print("=" * 50)
    print(f"Total Rows            : {len(df_merged)}")
    print(f"Unique Authors        : {df_merged['author'].nunique()}")
    
    print("\nDistribution by Source:")
    source_dist = df_merged["source"].value_counts()
    for src, count in source_dist.items():
        print(f"  • {src:<30}: {count}")
        
    print("\nTop 30 Authors by Article Count:")
    top_authors = df_merged["author"].value_counts().head(30)
    for auth, count in top_authors.items():
        print(f"  • {auth:<30}: {count}")
    print("=" * 50)

if __name__ == "__main__":
    main()


Starting StyloGuard Dataset Merging and Cleaning...
📖 Reading detik.csv from /kaggle/input/datasets/xandertrevor/stylometry/raw/detik.csv...
📖 Reading kumparan.csv from /kaggle/input/datasets/xandertrevor/stylometry/raw/kumparan.csv...
📖 Reading tempo.csv from /kaggle/input/datasets/xandertrevor/stylometry/raw/tempo.csv...
📖 Reading mojok.csv from /kaggle/input/datasets/xandertrevor/stylometry/raw/mojok.csv...
📖 Reading tirto.csv from /kaggle/input/datasets/xandertrevor/stylometry/raw/tirto.csv...
📖 Reading synthetic_imposter_dataset.csv from /kaggle/input/datasets/xandertrevor/stylometry/raw/synthetic_imposter_dataset.csv...

Initial merged rows: 6683
Rows after dropping explicit organizations: 6250
Rows after (author, text) deduplication: 4162
Rows after filtering low-frequency single-word authors: 4116

✅ Success! Merged dataset saved as 'merged_dataset.csv'

             MERGED DATASET SUMMARY
Total Rows            : 4116
Unique Authors        : 453

Distribution by Source:
  • det